# OD Model Benchmark

Comparison pipeline: **our GPS-based model** (pre-trained in `gps_od.ipynb`) vs
paper baselines and classical/graph/diffusion baselines.

**Our model:** GPS-GNN encoder + decoder, loaded from pre-trained weights.
**Benchmark baselines:** training and inference are separated in this notebook.
Train baseline artifacts once, then compare them against GPS via load-only inference.

In [ ]:
import pandas as pd

from benchmarking.config import (
    BASELINE_MODELS,
    DATA_PATH,
    GMEL_GPS_BENCHMARK_IDS,
    GPS_BENCHMARK_IDS,
    GPS_MC_BENCHMARK_IDS,
    INFERENCE_SEEDS,
    MULTI_CITY_IDS,
    PROJECT_ROOT,
    SINGLE_CITY_IDS,
    TRANSFLOWER_ORIG_CONFIG,
    WEIGHTS_DIR,
    cleanup_gpu,
    device,
    set_global_seed,
    trained_gps_run_ids,
    trained_multi_city_baseline_models,
    trained_single_city_baseline_models,
    trained_single_city_gps_run_ids,
    trained_single_city_lgbm_base_ids,
)
from benchmarking.data_utils import get_all_areas
from benchmarking.gps_loader import GPSBenchmarkLoader
from benchmarking.pipeline import (
    run_multi_city_benchmark,
    run_single_city_benchmark,
    train_multi_city_benchmark_models,
    train_single_city_benchmark_models,
)
from benchmarking.reporting import (
    build_combined_summary,
    plot_comparison,
    results_to_dataframe,
    save_results_table,
)

set_global_seed()

trained_sc = trained_single_city_gps_run_ids(GPS_BENCHMARK_IDS, SINGLE_CITY_IDS)
trained_mc = trained_gps_run_ids(GPS_MC_BENCHMARK_IDS)
LGBM_BENCHMARK_IDS = trained_single_city_lgbm_base_ids(GPS_BENCHMARK_IDS, SINGLE_CITY_IDS)
gmel_gps_run_ids = trained_single_city_gps_run_ids(GMEL_GPS_BENCHMARK_IDS, SINGLE_CITY_IDS)
trained_sc_baselines = trained_single_city_baseline_models(BASELINE_MODELS, SINGLE_CITY_IDS)
trained_mc_baselines = trained_multi_city_baseline_models(BASELINE_MODELS)

single_city_baseline_train_runs = {}
multi_city_baseline_train_runs = {}
single_city_results = {}
single_city_model_type = {}
multi_city_results = {}
multi_city_model_type = {}
multi_city_split = None
multi_city_training_split = None

gps_loader = GPSBenchmarkLoader(
    single_city_id=SINGLE_CITY_IDS[0],
    multi_city_ids=MULTI_CITY_IDS,
    data_path=DATA_PATH,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")
print(f"Data path: {DATA_PATH}")
print(f"Weights dir: {WEIGHTS_DIR}")

## Configurable Model List
Comment/uncomment to select which models to run.

In [ ]:
# Optional local override for notebook runs.
# BASELINE_MODELS = ["DGM", "GMEL", "GMEL_LGBM"]

print(f"Active baseline models: {BASELINE_MODELS}")

In [ ]:
print(f"GPS (SC) run_ids: {len(GPS_BENCHMARK_IDS)}  ({len(trained_sc)} fully trained across {len(SINGLE_CITY_IDS)} cities)")
print(f"GPS (MC) run_ids: {len(GPS_MC_BENCHMARK_IDS)}  ({len(trained_mc)} trained)")
print(f"LGBM donor ids:  {len(LGBM_BENCHMARK_IDS)}")
print(f"GMEL_GPS ids:    {len(gmel_gps_run_ids)}")
print(f"SC cities:       {SINGLE_CITY_IDS}")
print(f"Inference seeds: {INFERENCE_SEEDS}")
print(f"Baseline models: {len(BASELINE_MODELS)}")
print(f"SC baseline artifacts ready: {len(trained_sc_baselines)} / {len(BASELINE_MODELS)}")
print(f"MC baseline artifacts ready: {len(trained_mc_baselines)} / {len(BASELINE_MODELS)}")
print(f"SC baseline ready ids: {trained_sc_baselines}")
print(f"MC baseline ready ids: {trained_mc_baselines}")
print(f"TransFlower orig config: {TRANSFLOWER_ORIG_CONFIG.describe()}")

## Shared Data Utilities

In [ ]:
print(f"Total areas in dataset: {len(get_all_areas(DATA_PATH))}")

## Single-City Benchmark

All models predict on the cities in `SINGLE_CITY_IDS`.
Training and inference are separated:
1. train/save baseline artifacts per city
2. run inference-only comparison against pre-trained GPS variants

In [ ]:
# Single-city baseline training helper, split by benchmark city like in gps_od.ipynb
SC_CITY_1, SC_CITY_2, SC_CITY_3 = SINGLE_CITY_IDS
print(f"Single-city benchmark cities: {SINGLE_CITY_IDS}")

def train_sc_baseline_city(area_id):
    print()
    print("=" * 70)
    print(f"  SINGLE-CITY BASELINE TRAINING FOR {area_id}")
    print("=" * 70)
    city_runs = train_single_city_benchmark_models(
        single_city_ids=[area_id],
        data_path=DATA_PATH,
        baseline_models=BASELINE_MODELS,
        gps_loader=gps_loader,
    )
    single_city_baseline_train_runs[area_id] = city_runs
    trained_city_baselines = trained_single_city_baseline_models(BASELINE_MODELS, [area_id])
    print()
    print(f"Artifacts ready for {area_id}: {trained_city_baselines}")
    print(f"Training calls completed for {area_id}: {list(city_runs)}")
    return city_runs

In [ ]:
sc_city_1_baseline_runs = train_sc_baseline_city(SC_CITY_1)

In [ ]:
sc_city_2_baseline_runs = train_sc_baseline_city(SC_CITY_2)

In [ ]:
sc_city_3_baseline_runs = train_sc_baseline_city(SC_CITY_3)

In [ ]:
trained_sc_baselines = trained_single_city_baseline_models(BASELINE_MODELS, SINGLE_CITY_IDS)
print()
print(f"Artifacts ready across {SINGLE_CITY_IDS}: {trained_sc_baselines}")
print(f"Per-city training calls completed for: {list(single_city_baseline_train_runs)}")

In [ ]:
print("=" * 70)
print("  SINGLE-CITY BENCHMARK INFERENCE")
print(f"  Cities: {SINGLE_CITY_IDS}")
print(f"  Inference seeds: {INFERENCE_SEEDS}")
print("=" * 70)

trained_sc_baselines = trained_single_city_baseline_models(BASELINE_MODELS, SINGLE_CITY_IDS)
print(f"Baseline artifacts available: {trained_sc_baselines}")

single_city_results, single_city_model_type = run_single_city_benchmark(
    gps_run_ids=GPS_BENCHMARK_IDS,
    lgbm_run_ids=LGBM_BENCHMARK_IDS,
    single_city_ids=SINGLE_CITY_IDS,
    data_path=DATA_PATH,
    baseline_models=BASELINE_MODELS,
    gps_loader=gps_loader,
    gmel_gps_run_ids=gmel_gps_run_ids,
    inference_seeds=INFERENCE_SEEDS,
)

print()
print(f"Completed: {len(single_city_results)} models")

In [ ]:
if single_city_results:
    df_single = results_to_dataframe(single_city_results, single_city_model_type)

    print()
    print("=" * 100)
    print("  SINGLE-CITY RESULTS (3 cities, mean/std, sorted by CPC)")
    print("=" * 100)
    print(df_single.to_string())

    single_city_path = save_results_table(df_single, "single_city_benchmark.csv")
    print()
    print(f"Saved to {single_city_path}")
else:
    print("No results to display.")

## Multi-City (10 Cities) Benchmark

10 cities, area-level split: 8 train, 1 val, 1 test.
Training and inference are separated here as well:
1. train/save multi-city baseline artifacts
2. run inference-only comparison against pre-trained GPS variants

In [ ]:
# Run this cell when you need to (re)train and save multi-city baseline artifacts.
print("=" * 70)
print("  MULTI-CITY BASELINE TRAINING")
print(f"  Cities: {MULTI_CITY_IDS}")
print(f"  Baselines: {BASELINE_MODELS}")
print("=" * 70)

multi_city_baseline_train_runs, multi_city_training_split = train_multi_city_benchmark_models(
    city_ids=MULTI_CITY_IDS,
    data_path=DATA_PATH,
    baseline_models=BASELINE_MODELS,
    gps_loader=gps_loader,
)

trained_mc_baselines = trained_multi_city_baseline_models(BASELINE_MODELS)
print()
print(f"Artifacts ready for {len(trained_mc_baselines)} baseline models: {trained_mc_baselines}")
print(f"Training calls completed for: {list(multi_city_baseline_train_runs)}")
print(f"Train: {multi_city_training_split['train']}")
print(f"Valid: {multi_city_training_split['valid']}")
print(f"Test:  {multi_city_training_split['test']}")

In [ ]:
print("=" * 70)
print("  MULTI-CITY BENCHMARK INFERENCE")
print(f"  Cities: {MULTI_CITY_IDS}")
print(f"  Inference seeds: {INFERENCE_SEEDS}")
print("=" * 70)

trained_mc_baselines = trained_multi_city_baseline_models(BASELINE_MODELS)
print(f"Baseline artifacts available: {trained_mc_baselines}")

multi_city_results, multi_city_model_type, multi_city_split = run_multi_city_benchmark(
    gps_run_ids=GPS_MC_BENCHMARK_IDS,
    city_ids=MULTI_CITY_IDS,
    data_path=DATA_PATH,
    baseline_models=BASELINE_MODELS,
    gps_loader=gps_loader,
    inference_seeds=INFERENCE_SEEDS,
)

print(f"Train: {multi_city_split['train']}")
print(f"Valid: {multi_city_split['valid']}")
print(f"Test:  {multi_city_split['test']}")
print()
print(f"Completed: {len(multi_city_results)} models")

In [ ]:
if multi_city_results:
    df_multi = results_to_dataframe(multi_city_results, multi_city_model_type)

    print()
    print("=" * 100)
    print("  MULTI-CITY RESULTS (mean/std, sorted by CPC)")
    print("=" * 100)
    print(df_multi.to_string())

    multi_city_path = save_results_table(df_multi, "multi_city_benchmark.csv")
    print()
    print(f"Saved to {multi_city_path}")
else:
    print("No results to display.")

## Visualization

In [ ]:
if single_city_results:
    plot_comparison(single_city_results, f"Single-City Benchmark ({', '.join(SINGLE_CITY_IDS)})", ["CPC", "MAE", "RMSE"])
if multi_city_results:
    plot_comparison(multi_city_results, "Multi-City Benchmark (10 cities)", ["CPC", "MAE", "RMSE"])

In [ ]:
if single_city_results and multi_city_results:
    print()
    print("=" * 120)
    print("  COMBINED SUMMARY")
    print("=" * 120)

    df_summary = build_combined_summary(single_city_results, multi_city_results)
    print(df_summary.to_string())

    summary_path = save_results_table(df_summary, "benchmark_combined.csv")
    print()
    print(f"Saved to {summary_path}")
else:
    print("Run both benchmark inference cells first.")